In [1]:
import asyncio
import lsst_efd_client

import astropy.units as u
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import pytz
import sys
from astropy.stats import median_absolute_deviation as mad_astropy
from scipy.stats import binned_statistic
from astropy.time import Time
from tqdm import tqdm

from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
from lsst.summit.utils.efdUtils import makeEfdClient
import os
import pickle

%matplotlib inline

In [2]:
os.environ["no_proxy"] += ",.consdb"
consdb_url = 'http://consdb-pq.consdb:8080/consdb'
cdb_client = ConsDbClient(consdb_url)

efd_client = makeEfdClient()

In [3]:
# Get the focal plane positions of the corner wavefront sensors
from lsst.obs.lsst import LsstCam
import lsst.afw.cameraGeom as cameraGeom

camera = LsstCam.getCamera()

# Corner wavefront sensor detectors
CORNER_DETECTORS = [191, 192, 195, 196, 199, 200, 203, 204]

def get_detector_center_fp(det_id):
    """Get the center of a detector in focal plane coordinates (mm)."""
    det = camera[det_id]
    bbox = det.getBBox()
    center_x = (bbox.getMinX() + bbox.getMaxX()) / 2.0
    center_y = (bbox.getMinY() + bbox.getMaxY()) / 2.0
    
    tx = det.getTransform(cameraGeom.PIXELS, cameraGeom.FOCAL_PLANE)
    fpx, fpy = tx.getMapping().applyForward(np.array([[center_x], [center_y]], dtype=float))
    return float(fpx), float(fpy)

# Build dictionary of detector positions
DETECTOR_FP_POSITIONS = {}
for det_id in CORNER_DETECTORS:
    fpx, fpy = get_detector_center_fp(det_id)
    DETECTOR_FP_POSITIONS[det_id] = {'fpx': fpx, 'fpy': fpy, 'name': camera[det_id].getName()}
    print(f"Detector {det_id} ({camera[det_id].getName()}): fpx={fpx:.1f} mm, fpy={fpy:.1f} mm")

Detector 191 (R00_SW0): fpx=-214.2 mm, fpy=-202.6 mm
Detector 192 (R00_SW1): fpx=-214.3 mm, fpy=-225.9 mm
Detector 195 (R04_SW0): fpx=202.6 mm, fpy=-214.2 mm
Detector 196 (R04_SW1): fpx=225.9 mm, fpy=-214.3 mm
Detector 199 (R40_SW0): fpx=-202.6 mm, fpy=214.2 mm
Detector 200 (R40_SW1): fpx=-225.9 mm, fpy=214.3 mm
Detector 203 (R44_SW0): fpx=214.2 mm, fpy=202.6 mm
Detector 204 (R44_SW1): fpx=214.3 mm, fpy=225.9 mm


/tmp/ipykernel_2626/3909363133.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(fpx), float(fpy)


In [4]:
query = f"""
SELECT
e.exposure_id AS visit_id,
e.band, 
e.day_obs AS day_obs,
e.obs_start_mjd,
e.obs_end_mjd,
e.exp_midpt AS time,
e.dimm_seeing AS seeing,
e.focus_z AS focus_z, 
e.science_program AS science_program,
e.wind_speed AS wind_speed,
e.wind_dir AS wind_dir,
ccdvisit1_quicklook.psf_sigma,
ccdvisit1_quicklook.z4,
ccdvisit1_quicklook.z5,
ccdvisit1_quicklook.z6,
ccdvisit1_quicklook.z7,
ccdvisit1_quicklook.z8,
ccdvisit1_quicklook.z9,
ccdvisit1_quicklook.z10,
ccdvisit1_quicklook.z11,
ccdvisit1_quicklook.z12,
ccdvisit1_quicklook.z13,
ccdvisit1_quicklook.z14,
ccdvisit1_quicklook.z15,
ccdvisit1_quicklook.z16,
ccdvisit1_quicklook.z17,
ccdvisit1_quicklook.z18,
ccdvisit1_quicklook.z19,
ccdvisit1_quicklook.z20,
ccdvisit1_quicklook.z21,
ccdvisit1_quicklook.z22,
ccdvisit1_quicklook.z23,
ccdvisit1_quicklook.z24,
ccdvisit1_quicklook.z25,
ccdvisit1_quicklook.z26,
ccdvisit1_quicklook.z27,
ccdvisit1_quicklook.z28,
ccdvisit1.detector as detector,
q.donut_blur_fwhm AS donut_blur,
q.ringss_seeing AS ringss_seeing,
q.psf_sigma_median AS psf_fwhm,
q.psf_sigma_min AS psf_fwhm_min,
q.psf_sigma_max AS psf_fwhm_max,
q.physical_rotator_angle,
e.obs_end,
e.obs_start,
e.seq_num
FROM
cdb_lsstcam.ccdvisit1_quicklook AS ccdvisit1_quicklook,
cdb_lsstcam.ccdvisit1 AS ccdvisit1,
cdb_lsstcam.visit1 AS visit1,
cdb_lsstcam.visit1_quicklook AS q,
cdb_lsstcam.exposure AS e
WHERE
ccdvisit1.detector IN (191, 192, 195, 196, 199, 200, 203, 204)
AND ccdvisit1.ccdvisit_id = ccdvisit1_quicklook.ccdvisit_id
AND ccdvisit1.visit_id = visit1.visit_id
AND ccdvisit1.visit_id = q.visit_id
AND ccdvisit1.visit_id = e.exposure_id
AND (e.img_type = 'science')
AND e.airmass > 0
AND e.band != 'none'
"""

table = cdb_client.query(query).to_pandas()

In [5]:
# Original format (for backward compatibility)
zernike4 = {}

for i in range(len(table)):
    if int(table['visit_id'][i]) not in zernike4:
        zernike4.update({int(table['visit_id'][i]): {
            "visit_id": table['visit_id'][i],
            "band": table['band'][i],
            "obs_start_mjd": table['obs_start_mjd'][i],
            "obs_end_mjd": table["obs_end_mjd"][i],
        }})
        for j in range(28-4):
            zcoeff = table['z%i'%(j+4)][i]
            if zcoeff is None:
                zcoeff = np.nan
            else:
                zcoeff = float(zcoeff)
            zernike4[int(table['visit_id'][i])].update({"z%i"%(j+4): [zcoeff],})
            
    else:
        for j in range(28-4):
            zcoeff = table['z%i'%(j+4)][i]
            if zcoeff is None:
                zcoeff = np.nan
            else:
                zcoeff = float(zcoeff)
            zernike4[int(table['visit_id'][i])]["z%i"%(j+4)].append(zcoeff)

visitIDs = [visit for visit in zernike4]
visitIDs.sort()

# Save original format
filePkl = open('data/visit_to_band_mapv3.pkl', 'wb')
pickle.dump(zernike4, filePkl)
filePkl.close()
print(f"Saved original format to data/visit_to_band_mapv2.pkl ({len(zernike4)} visits)")

Saved original format to data/visit_to_band_mapv2.pkl (39742 visits)


In [6]:
# New format with detector positions for corner plotting
# Structure: {visit_id: {z4_corners: [{detector: det_id, fpx: x, fpy: y, z4: value}, ...], ...}}

zernike_corners = {}

for i in range(len(table)):
    visit_id = int(table['visit_id'][i])
    detector = int(table['detector'][i])
    
    if visit_id not in zernike_corners:
        zernike_corners[visit_id] = {
            "visit_id": table['visit_id'][i],
            "band": table['band'][i],
            "obs_start_mjd": table['obs_start_mjd'][i],
            "obs_end_mjd": table["obs_end_mjd"][i],
        }
        # Initialize corner lists for each zernike
        for j in range(28-4):
            zernike_corners[visit_id][f"z{j+4}_corners"] = []
    
    # Get detector position
    if detector in DETECTOR_FP_POSITIONS:
        det_info = DETECTOR_FP_POSITIONS[detector]
        
        # Add corner measurements for each zernike
        for j in range(28-4):
            zcoeff = table[f'z{j+4}'][i]
            if zcoeff is None:
                zcoeff = np.nan
            else:
                zcoeff = float(zcoeff)
            
            zernike_corners[visit_id][f"z{j+4}_corners"].append({
                'detector': detector,
                'det_name': det_info['name'],
                'fpx': det_info['fpx'],
                'fpy': det_info['fpy'],
                'value': zcoeff,
            })

# Save new format with corners
filePkl = open('data/visit_zernike_corners.pkl', 'wb')
pickle.dump(zernike_corners, filePkl)
filePkl.close()
print(f"Saved corner format to data/visit_zernike_corners.pkl ({len(zernike_corners)} visits)")

Saved corner format to data/visit_zernike_corners.pkl (39742 visits)


In [ ]:
# Verify the new format
test_visit = visitIDs[0]
print(f"Example visit {test_visit}:")
print(f"  Band: {zernike_corners[test_visit]['band']}")
print(f"  Number of z4 corner measurements: {len(zernike_corners[test_visit]['z4_corners'])}")
print(f"  z4 corners:")
for corner in zernike_corners[test_visit]['z4_corners']:
    print(f"    Detector {corner['detector']} ({corner['det_name']}): "
          f"fpx={corner['fpx']:.1f}, fpy={corner['fpy']:.1f}, z4={corner['value']:.4f}")

In [ ]:
# Test plot: z4 at corners for a single visit
test_visit = visitIDs[100]  # Pick a visit

fig, ax = plt.subplots(figsize=(10, 10))

z4_corners = zernike_corners[test_visit]['z4_corners']
fpx = [c['fpx'] for c in z4_corners]
fpy = [c['fpy'] for c in z4_corners]
z4_vals = [c['value'] for c in z4_corners]

sc = ax.scatter(fpx, fpy, c=z4_vals, s=500, cmap='seismic', 
                vmin=-0.5, vmax=0.5, edgecolor='black', linewidth=2)

# Add labels
for corner in z4_corners:
    ax.annotate(f"{corner['det_name']}\nz4={corner['value']:.3f}", 
                (corner['fpx'], corner['fpy']),
                textcoords="offset points", xytext=(0, 30),
                ha='center', fontsize=10)

cb = plt.colorbar(sc)
cb.set_label('z4 (μm)', fontsize=14)
ax.set_xlabel('x (mm)', fontsize=14)
ax.set_ylabel('y (mm)', fontsize=14)
ax.set_title(f"Visit {test_visit} | z4 at corner wavefront sensors", fontsize=14)
ax.set_aspect('equal')
ax.set_xlim(-350, 350)
ax.set_ylim(-350, 350)
plt.tight_layout()
plt.show()